In [4]:
import duckdb
from pathlib import Path

# Path to the DuckDB database file
DATA_DIR = Path("../data/raw")

# Connect to the DuckDB database
con = duckdb.connect("../northwind.duckdb")

# Task 04: Revenue Foundation

## Goal

Calculate revenue correctly from `order_details`.

## What you will learn

```text
gross revenue
discount amount
net revenue
line-level calculation
order-level revenue
SQL views
```

## Main formulas

```text
gross_revenue = unitPrice * quantity
discount_amount = unitPrice * quantity * discount
net_revenue = unitPrice * quantity * (1 - discount)
```

## Step 1: Inspect revenue inputs

```python
display(con.sql("""
    SELECT COUNT(*) AS total_rows,
           MIN(unitPrice) AS min_unit_price,
           MAX(unitPrice) AS max_unit_price,
           MIN(quantity) AS min_quantity,
           MAX(quantity) AS max_quantity,
           MIN(discount) AS min_discount,
           MAX(discount) AS max_discount
    FROM order_details;
""").df())
```

In [8]:
display(con.sql("""
    SELECT COUNT(*) AS total_rows,
           MIN(unitPrice) AS min_unit_price,
           MAX(unitPrice) AS max_unit_price,
           MIN(quantity) AS min_quantity,
           MAX(quantity) AS max_quantity,
           MIN(discount) AS min_discount,
           MAX(discount) AS max_discount
    FROM order_details;
""").df())

,total_rows,min_unit_price,max_unit_price,min_quantity,max_quantity,min_discount,max_discount
0,2155,2.0,263.5,1,130,0.0,0.25


## Step 2: Check bad revenue inputs

```python
display(con.sql("""
    SELECT *
    FROM order_details
    WHERE unitPrice IS NULL
       OR quantity IS NULL
       OR discount IS NULL
       OR unitPrice <= 0
       OR quantity <= 0
       OR discount < 0
       OR discount > 1;
""").df())
```

In [11]:
con.sql("""
SELECT COUNT(*) AS Bad_rows 
FROM order_details
WHERE unitPrice <=0 OR quantity<=0 OR discount<0 OR discount >1 OR unitPrice IS NULL OR quantity IS NULL OR discount IS NULL;        

""").df()

,Bad_rows
0,0


Empty result means revenue inputs look safe.

## Step 3: Calculate gross revenue

```python
display(con.sql("""
    SELECT orderID, productID, unitPrice, quantity, discount,
           unitPrice * quantity AS gross_revenue
    FROM order_details
    LIMIT 10;
""").df())
```

In [ ]:
display(con.sql("""
    SELECT orderID, productID, unitPrice, quantity, discount,
           unitPrice * quantity AS gross_revenue
    FROM order_details
""").df())

,orderID,productID,unitPrice,quantity,discount,gross_revenue
0,10248,11,14.00,12,0.00,168.0
1,10248,42,9.80,10,0.00,98.0
2,10248,72,34.80,5,0.00,174.0
3,10249,14,18.60,9,0.00,167.4
4,10249,51,42.40,40,0.00,1696.0
...,...,...,...,...,...,...
2150,11077,64,33.25,2,0.03,66.5
2151,11077,66,17.00,1,0.00,17.0
2152,11077,73,15.00,2,0.01,30.0
2153,11077,75,7.75,4,0.00,31.0



## Step 4: Calculate discount amount

```python
display(con.sql("""
    SELECT orderID, productID, unitPrice, quantity, discount,
           unitPrice * quantity AS gross_revenue,
           unitPrice * quantity * discount AS discount_amount
    FROM order_details
    LIMIT 10;
""").df())
```

In [7]:
con.sql("""SELECT orderID, productID, unitPrice, quantity, discount,
        unitPrice * quantity * discount AS discount_amount
FROM order_details
        LIMIT 10""")

┌─────────┬───────────┬───────────┬──────────┬──────────┬────────────────────┐
│ orderID │ productID │ unitPrice │ quantity │ discount │  discount_amount   │
│  int64  │   int64   │  double   │  int64   │  double  │       double       │
├─────────┼───────────┼───────────┼──────────┼──────────┼────────────────────┤
│   10248 │        11 │      14.0 │       12 │      0.0 │                0.0 │
│   10248 │        42 │       9.8 │       10 │      0.0 │                0.0 │
│   10248 │        72 │      34.8 │        5 │      0.0 │                0.0 │
│   10249 │        14 │      18.6 │        9 │      0.0 │                0.0 │
│   10249 │        51 │      42.4 │       40 │      0.0 │                0.0 │
│   10250 │        41 │       7.7 │       10 │      0.0 │                0.0 │
│   10250 │        51 │      42.4 │       35 │     0.15 │              222.6 │
│   10250 │        65 │      16.8 │       15 │     0.15 │               37.8 │
│   10251 │        22 │      16.8 │        6 │     0


## Step 5: Calculate net revenue

```python
display(con.sql("""
    SELECT orderID, productID, unitPrice, quantity, discount,
           unitPrice * quantity AS gross_revenue,
           unitPrice * quantity * discount AS discount_amount,
           unitPrice * quantity * (1 - discount) AS net_revenue
    FROM order_details
    LIMIT 10;
""").df())
```

## Step 6: Total net revenue

```python
display(con.sql("""
    SELECT ROUND(SUM(unitPrice * quantity * (1 - discount)), 2) AS total_net_revenue
    FROM order_details;
""").df())
```

In [11]:
display(con.sql("""
    SELECT orderID, productID, unitPrice, quantity, discount,
           unitPrice * quantity AS gross_revenue,
           unitPrice * quantity * discount AS discount_amount,
           unitPrice * quantity * (1 - discount) AS net_revenue
    FROM order_details
    LIMIT 10;
""").df())

display(con.sql("""
    SELECT ROUND(SUM(unitPrice*quantity*(1-discount)),2) AS total_net_revenue
    FROM order_details;
""").df())



,orderID,productID,unitPrice,quantity,discount,gross_revenue,discount_amount,net_revenue
0,10248,11,14.0,12,0.00,168.0,0.00,168.00
1,10248,42,9.8,10,0.00,98.0,0.00,98.00
2,10248,72,34.8,5,0.00,174.0,0.00,174.00
3,10249,14,18.6,9,0.00,167.4,0.00,167.40
4,10249,51,42.4,40,0.00,1696.0,0.00,1696.00
5,10250,41,7.7,10,0.00,77.0,0.00,77.00
6,10250,51,42.4,35,0.15,1484.0,222.60,1261.40
7,10250,65,16.8,15,0.15,252.0,37.80,214.20
8,10251,22,16.8,6,0.05,100.8,5.04,95.76
9,10251,57,15.6,15,0.05,234.0,11.70,222.30


,total_net_revenue
0,1265793.04



## Step 7: Revenue per order

```python
display(con.sql("""
    SELECT orderID,
           ROUND(SUM(unitPrice * quantity * (1 - discount)), 2) AS order_revenue,
           COUNT(productID) AS product_lines,
           SUM(quantity) AS total_units
    FROM order_details
    GROUP BY orderID
    ORDER BY order_revenue DESC;
""").df())
```

## Step 8: Create sales_lines view

```python
con.sql("""
CREATE OR REPLACE VIEW sales_lines AS
SELECT od.orderID,
       od.productID,
       od.unitPrice,
       od.quantity,
       od.discount,
       od.unitPrice * od.quantity AS gross_revenue,
       od.unitPrice * od.quantity * od.discount AS discount_amount,
       od.unitPrice * od.quantity * (1 - od.discount) AS net_revenue
FROM order_details AS od;
""")
```

Test it:

```python
display(con.sql("SELECT * FROM sales_lines LIMIT 10;").df())
```

## Step 9: Validate sales_lines

```python
display(con.sql("""
    SELECT COUNT(*) AS sales_line_rows,
           ROUND(SUM(net_revenue), 2) AS total_net_revenue
    FROM sales_lines;
""").df())
```

```python
display(con.sql("SELECT COUNT(*) AS order_detail_rows FROM order_details;").df())
```

In [9]:
display(con.sql("""
SELECT orderID, ROUND(SUM(unitPrice*quantity*(1-discount)),2) AS net_revenue, SUM(quantity) AS total_quantity
FROM order_details
GROUP BY orderID
""").df())

con.sql("""
CREATE OR REPLACE VIEW sales_lines AS
SELECT od.orderID,
       od.productID,
       od.unitPrice,
       od.quantity,
       od.discount,
       od.unitPrice * od.quantity AS gross_revenue,
       od.unitPrice * od.quantity * od.discount AS discount_amount,
       od.unitPrice * od.quantity * (1 - od.discount) AS net_revenue
FROM order_details AS od;
""")

display(con.sql("SELECT * FROM sales_lines ORDER BY net_revenue DESC LIMIT 10;").df())


,orderID,net_revenue,total_quantity
0,10248,440.00,27.0
1,10249,1863.40,49.0
2,10250,1552.60,60.0
3,10251,654.06,41.0
4,10252,3597.90,105.0
...,...,...,...
825,11073,300.00,30.0
826,11074,232.08,14.0
827,11075,498.10,42.0
828,11076,792.75,50.0


,orderID,productID,unitPrice,quantity,discount,gross_revenue,discount_amount,net_revenue
0,10981,38,263.50,60,0.00,15810.0,0.00,15810.00
1,10865,38,263.50,60,0.05,15810.0,790.50,15019.50
2,10417,38,210.80,50,0.00,10540.0,0.00,10540.00
3,10889,38,263.50,40,0.00,10540.0,0.00,10540.00
4,10897,29,123.79,80,0.00,9903.2,0.00,9903.20
5,10353,38,210.80,50,0.20,10540.0,2108.00,8432.00
6,10424,38,210.80,49,0.20,10329.2,2065.84,8263.36
7,10540,38,263.50,30,0.00,7905.0,0.00,7905.00
8,10817,38,263.50,30,0.00,7905.0,0.00,7905.00
9,10816,38,263.50,30,0.05,7905.0,395.25,7509.75



## Observation template

```text
Revenue input summary:
Bad revenue inputs found:
Total net revenue:
Highest revenue order:
Difference between product_lines and total_units:
Was sales_lines view created:
Why sales_lines is useful:
```

## Stop point

You are done when you understand that revenue comes from `order_details`, must include `quantity`, and must include `discount`.